In [2]:
import json
import numpy as np
import os
import re
def remove_references(text):
    
    # 处理各种引用格式
    # 1. 单个或多个作者(包括et al.)加年份的组合
    text = re.sub(r'\(\s*(?:[A-Za-z\s\-\.]+(?:\s+et\s+al\.)?(?:\s*;\s*[A-Za-z\s\-\.]+(?:\s+et\s+al\.)?)*),\s*\d{4}(?:,\s*\d{4})*\s*\)', '', text)
    
    # 2. 处理作者名称可能包含"and"或"&"的情况
    text = re.sub(r'\(\s*(?:[A-Za-z\s\-\.]+(?:\s+(?:and|&)\s+[A-Za-z\s\-\.]+)(?:\s*;\s*[A-Za-z\s\-\.]+(?:\s+(?:and|&)\s+[A-Za-z\s\-\.]+)?)*),\s*\d{4}(?:,\s*\d{4})*\s*\)', '', text)
    
    # 3. 处理组织名称作为作者的情况 (如 Council of Europe)
    text = re.sub(r'\(\s*(?:[A-Za-z\s\-\.]+(?:\s+of\s+[A-Za-z\s\-\.]+)?),\s*\d{4}(?:,\s*\d{4})*\s*\)', '', text)
    
    # 4. 处理多个引用组合的情况 (作者1, 年份; 作者2, 年份)
    text = re.sub(r'\(\s*(?:[A-Za-z\s\-\.]+(?:\s+et\s+al\.)?),\s*\d{4}(?:,\s*\d{4})*(?:\s*;\s*[A-Za-z\s\-\.]+(?:\s+et\s+al\.)?),\s*\d{4}(?:,\s*\d{4})*\s*\)', '', text)
    
    # 5. 处理只有组织缩写的情况 (如 ALTE, 2005; 2011)
    text = re.sub(r'\(\s*[A-Z]+,\s*\d{4}(?:;\s*\d{4})*\s*\)', '', text)
    
    # 6. 处理任何剩余的标准引用格式
    text = re.sub(r'\(\s*[A-Za-z\s\-\.]+,\s*\d{4}(?:,\s*\d{4})*\s*\)', '', text)
    
    
    # 2. 常见学术引用的特定模式
    text = re.sub(r'\(\s*[A-Za-z\s\-\.]+(?:\s+et\s+al\.)?(?:\s*and\s*[A-Za-z\s\-\.]+)?,\s*\d{4}(?:[a-z])?(?:,\s*\d{4}(?:[a-z])?)*\s*\)', '', text)
    
    # 3. 用分号分隔的多作者模式
    text = re.sub(r'\(\s*(?:[A-Za-z\s\-\.]+(?:\s+et\s+al\.)?(?:,\s*\d{4}(?:[a-z])?)+(?:\s*;\s*[A-Za-z\s\-\.]+(?:\s+et\s+al\.)?(?:,\s*\d{4}(?:[a-z])?)+)*)\s*\)', '', text)
    
    # 4. 组织名称引用模式
    text = re.sub(r'\(\s*[A-Za-z][A-Za-z\s\-\.]+(?:\s+of\s+[A-Za-z\s\-\.]+)?,\s*\d{4}(?:[a-z])?(?:,\s*\d{4}(?:[a-z])?)*\s*\)', '', text)
    
    # 5. 缩写和首字母缩略词模式
    text = re.sub(r'\(\s*[A-Z][A-Za-z\s\-\.]*,\s*\d{4}(?:[a-z])?(?:,\s*\d{4}(?:[a-z])?)*\s*\)', '', text)
    
    # 6. 特定模式，处理类似(Wen et al., 2015a,b)的引用
    text = re.sub(r'\(\s*[A-Za-z\s\-\.]+(?:\s+et\s+al\.)?(?:,\s*\d{4}[a-z]?)+\s*\)', '', text)
    
    # 7. 处理多年份用分号分隔的引用
    text = re.sub(r'\(\s*[A-Za-z\s\-\.]+(?:\s+et\s+al\.)?(?:,\s*\d{4}(?:[a-z])?)+(?:\s*;\s*\d{4}(?:[a-z])?)+\s*\)', '', text)
    
    # 8. 处理包含特殊字符的作者名(如Dusˇek and Jurcˇı´cˇek, 2016)
    text = re.sub(r'\(\s*[A-Za-zˇ´\s\-\.]+(?:\s+and\s+[A-Za-zˇ´\s\-\.]+)?,\s*\d{4}(?:[a-z])?(?:,\s*\d{4}(?:[a-z])?)*\s*\)', '', text)
    return text

In [4]:
# 计算ppl
path = '../writing_agents_results_cvpr/gpt_naive_ppl.json'
with open(path,'r',encoding='utf-8') as fp:
    naive = json.load(fp)
print(np.mean([item if item<40 else 40 for item in list(naive.values())]))

28.566330591837566


In [88]:
# 计算 structure

In [5]:
def load_json(path):
    with open(path,'r',encoding='utf-8') as fp:
        data = json.load(fp)
    return data
def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as fp:
        json.dump(data, fp, ensure_ascii=False, indent=4)
base_dir = '../writing_agents_results_cvpr/gpt_elaborate_judge/'
base_files = [file for file in os.listdir(base_dir) if file.endswith('.json')]
base_data = {}
for file in base_files:
    path = base_dir + file
    try:
        with open(path, "r", encoding="utf-8") as fp:
            data = json.load(fp)
        base_data[file[:-5]] = data
    except:
        print(file)
len(base_data)

102

In [6]:
save_json(base_data, 'temp_structure.json')

In [28]:
# 计算 coverate
dd = load_json('../evaluate/evaluate_log/simi/qwen/qwen_outline_consistency.json')

In [29]:
import numpy as np
np.mean([item for item in list(dd.values()) if item>0])

np.float64(0.3334598326059112)

In [18]:
#  length
from data_load import *
import json
def load_data(data_dir):
    folders = [folder for folder in os.listdir(data_dir) if 'pdf' in folder]
    all_data = {}
    for folder in folders:
        path = data_dir + folder + '/' + 'introduction_results.json'
        if os.path.exists(path): 
            with open(path, 'r', encoding='utf-8') as fp:
                data = json.load(fp)
            if 'sections' in data.keys() and 'introduction' in data.keys():
                all_data[folder] = data
    return all_data
    
# loader = DataLoader("/home/mczhang/zmc-dl/LLM/NTP/paper_data/acl/2025/main")
loader = DataLoader("/home/mczhang/zmc-dl/LLM/NTP/paper_data/cvpr/2025")
ori_data = loader.load_all()
all_items = []
for item in ori_data.keys():
    if ori_data[item].abstract:
        all_items.append(item)
print(len(all_items))

102


In [19]:
import numpy as np
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, LogitsProcessor, LogitsProcessorList
)
truth_len = [len(ori_data[item].introduction.split()) for item in all_items]

In [26]:
model = load_data('../writing_agents_results_cvpr/naive_sft/')
# len(model)
model_len = [len(model[item]['introduction'].split()) for item in all_items]
np.mean([model_introduction[i] == remove_references(model_introduction[i]) for i in range(len(model_introduction))])

np.float64(1.0)

In [8]:
model_introduction = [model[item]['introduction'] for item in all_items]

In [13]:
model_path = "/home/tcsu/Qwen2.5-7B-Instruct/" 
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [23]:
np.mean([len(tokenizer(intro)['input_ids']) for intro in model_introduction])

NameError: name 'tokenizer' is not defined

In [9]:
print(model_introduction[1])

In the rapidly progressing field of natural language processing, the task of text summarization has consistently presented a compelling challenge. Text summarization aims to condense large volumes of information into concise, coherent, and contextually relevant snippets. Traditionally, evaluation frameworks for summarization models have focused predominantly on syntactic and semantic coherence, primarily conducted within English and other major languages. Recent advancements, however, have brought to attention the necessity for more nuanced evaluation frameworks that incorporate a broader range of linguistic and domain-specific contexts.

While the field has witnessed substantial methodological advancements, existing benchmarks remain constrained, both in terms of linguistic diversity and domain specificity. The conventional evaluation frameworks predominantly provide metrics that apply universally rather than adapting to the particularities of diverse domains, resulting in an apparent

In [21]:
np.mean([model_introduction[i] == remove_references(model_introduction[i]) for i in range(len(model_introduction))])

np.float64(1.0)

In [18]:
np.mean(model_len)

np.float64(491.84098639455783)

In [22]:
sum([abs(truth_len[i]-model_len[i])<=truth_len[i]*0.25 for i in range(len(model_len))])/len(truth_len)

0.6028911564625851

In [33]:
sum([model_len[i]>300 and model_len[i]<1000 for i in range(len(model_len))])

1068

In [46]:
outlines = load_json('../evaluate/outline_2025.json')
len(outlines)

1490

In [48]:
outlines['2025.acl-long.1.pdf-186f2ec4-267b-4f6b-9d91-50b82537d464'][sections]

{'sections': {'Background': 'In our daily lives, humans commonly plan a sequence of general prototypical actions, usually in the form of step-by-step instructions, to achieve a specific objective. This capability, also known as goal-oriented script planning, forms the foundation of situationally grounded planning for complex scenarios, which is crucial for intelligent agents. With recent advances in Large Language Models, recent works have demonstrated the strong capabilities of LLMs in script planning. This has led to the development of LLM-based script planners across downstream domains. In e-commerce, there is a growing trend of customers querying LLM-empowered shopping assistants to create scripts tailored to their specific needs or objectives, with each step featuring relevant products. A goal-oriented script with product recommendations at certain steps that may necessitate product purchases, which we term as product-enriched script, facilitates a convenient shopping experience, 

In [62]:
new_outlines = {}
ok_items = list(base_data.keys())
for item in ok_items:
    new_outlines[item] = {
        "sections":{
            "background":outlines[item]['sections']['Background'] if 'Background' in outlines[item]['sections'].keys() else '',
            "problem_statement":outlines[item]['sections']['Problem and Limitations of Existing Methods'] if 'Problem and Limitations of Existing Methods' in outlines[item]['sections'].keys() else '',
            "method_overview":outlines[item]['sections']['Brief Method Overview and Summary of Main Results'] if 'Brief Method Overview and Summary of Main Results' in outlines[item]['sections'].keys() else '',
            "contributions":outlines[item]['sections']['Our Contributions'] if 'Our Contributions' in outlines[item]['sections'].keys() else '',
        },
        "outline":{
            "background":outlines[item]['outline']['Background'] if 'Background' in outlines[item]['outline'].keys() else [],
            "problem_statement":outlines[item]['outline']['Problem and Limitations of Existing Methods'] if 'Problem and Limitations of Existing Methods' in outlines[item]['outline'].keys() else [],
            "method_overview":outlines[item]['outline']['Brief Method Overview and Summary of Main Results'] if 'Brief Method Overview and Summary of Main Results' in outlines[item]['outline'].keys() else [],
            "contributions":outlines[item]['outline']['Our Contributions'] if 'Our Contributions' in outlines[item]['outline'].keys() else [],
        },
        "introduction": outlines[item]['introduction']
    } 

In [64]:
def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as fp:
        json.dump(data, fp, ensure_ascii=False, indent=4)
save_json(new_outlines, '../evaluate/outline_acl_2025_main.json')

In [10]:
import numpy as np
sorted([2,3,4,5,6])[:3] + sorted([2,3,4,5,6])[:-4:-1]

[2, 3, 4, 6, 5, 4]

In [6]:
sa = {}
sections = ['Background','Problem','Method','Contributions']
for section in sections:
    sa[section] = ''
lis = ['Background','Method','Problem','Method','Contributions','Background','Method']
new_lis = []
now = 'Background'
for i in range(len(lis)):
    if i == 0:
        new_lis.append(lis[i])
    elif sections.index(lis[i])<=sections.index(now):
        new_lis.append(now)
    else:
        now = lis[i]
        new_lis.append(now)  
new_lis
    

['Background',
 'Method',
 'Method',
 'Method',
 'Contributions',
 'Contributions',
 'Contributions']

In [7]:
import json
def load_json(path):
    with open(path,'r',encoding='utf-8') as fp:
        data = json.load(fp)
    return data
data = load_json('../results/llama_outline_base_0913.json')

In [16]:
[item for item in data['2025.acl-long.262.pdf-6b1c42f8-0b3d-412a-9fe2-09d62ab4409b']['introduction'].split('\n') if item!='']
# for 

['Solving complex tasks in a single attempt poses significant challenges for large language models (LLMs), primarily due to their inherent limitations in knowledge acquisition and experiential background. These constraints necessitate iterative interactions with the environment and receipt of feedback to facilitate successful resolution. Consequently, optimizing the utility of received feedback emerges as a pivotal aspect in empowering LLMs to engage in informed reasoning and problem-solving processes efficiently. Despite the significance of feedback incorporation, existing methodologies frequently fall short in addressing the complexities associated with this endeavor. Specifically, prevalent strategies either grapple with length generalization or resort to simplistic retry mechanisms devoid of meaningful engagement with prior information.',
 'Moreover, current practices tend to favor expediency over precision, compromising the overall dependability and performance of the underlying s

In [10]:
qwen_naive_sft = load_data('../writing_agents_results_llama/llama_naive_sft/')

In [11]:
sections = ['background','problem_statement','method_overview','contributions']
c = 0 
lack_sec = []
for item in qwen_naive_sft.keys():
    lacking = 0
    for s in sections:
        if qwen_naive_sft[item]['sections'][s] == '':
            lacking += 1
            lack_sec.append(s)
    if lacking != 0:
        c += 1
        print(item)
c

2025.acl-long.1572.pdf-f9e4d961-b587-45c3-9b5b-5ec606296fff
2025.acl-long.289.pdf-3377617f-5c7c-4434-91fc-55910c4913e1
2025.acl-long.1570.pdf-0741fb76-d32d-46dd-bb06-6d4544cbb081
2025.acl-long.1008.pdf-ca74a286-63f7-4d38-a195-ad8d21a32075
2025.acl-long.1466.pdf-04f97c48-a6ce-45a0-81f2-a642f6ae2982
2025.acl-long.1062.pdf-7faf42b1-a313-482c-af4a-fa6821099c06
2025.acl-long.625.pdf-6c859ecc-f31e-4e13-81d4-c309d2908c12
2025.acl-long.1204.pdf-28175f90-616a-4f95-8915-faf8dcba314a
2025.acl-long.1409.pdf-d50abf41-e214-4eea-8340-f18d7de47c67
2025.acl-long.847.pdf-22c613d4-aaed-4b1d-95cd-f54b13d6da51
2025.acl-long.979.pdf-96c4b45e-ade6-4d78-8aaa-7f309b2ca1fb
2025.acl-long.794.pdf-8570bd60-d774-4842-9d00-dee903bbe2ac
2025.acl-long.1152.pdf-866364cc-d278-4bf7-9ac5-fa8878596501
2025.acl-long.366.pdf-29735736-709b-406a-8bf6-2c2a02ad59e8
2025.acl-long.901.pdf-71bf6102-69af-4faf-8869-e9d2a7404f4a
2025.acl-long.362.pdf-6db0e664-3a4e-4f34-9ce4-4723183773db
2025.acl-long.483.pdf-f4c08316-b120-4ae8-ad87-09

285

In [15]:
sum([s==sections[3] for s in lack_sec])

113